In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from collections import OrderedDict
from datasets import load_dataset
from torch.utils.data import DataLoader
from scipy.sparse import csr_matrix
from sklearn.metrics import classification_report
import torch.nn as nn

import experiment
import clslib
import analysis
import smlp

In [2]:
dataset = load_dataset("stanfordnlp/sst2")
# temp_train_df = dataset["train"].to_pandas()
# temp_val_df = dataset["validation"].to_pandas()
# dataset_df = pd.concat([temp_train_df, temp_val_df], ignore_index=True).drop(columns=['idx'])
# train_df, val_df, test_df = np.split(dataset_df, [int(0.8 * len(dataset_df)), int(0.9 * len(dataset_df))])

# train_dataset = Dataset.from_pandas(train_df)
# val_dataset = Dataset.from_pandas(val_df)
# test_dataset = Dataset.from_pandas(test_df)
train_dataset = dataset["train"]
test_dataset = dataset["validation"]

In [3]:
vocab_size = 1000
min_frequency = 2
my_seeds = [0]
num_folds = 5
my_cross_validation_sets = []
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)

for seed in my_seeds:
    my_cross_validation_sets.append(
        experiment.cross_validation_split(
            pairs=list(zip(train_dataset["sentence"], train_dataset["label"])),
            k=num_folds,
            seed=seed
        )
    )

bpe_tokenizer = experiment.train_bpe_tokenizer(train_dataset["sentence"], vocab_size, min_frequency)

tfidf_encoder = clslib.TFIDF(bpe_tokenizer)

my_classifiers = OrderedDict()

In [ ]:
hidden_dim=128
dropout=0.1
batch_size=64
learning_rate=1e-3
weight_decay=1e-4
num_epochs=10
device='cpu'

my_classifiers['singleMLP_tfidf-enc'] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/singleMLP_tfidf-enc/seed-{seed}"

    my_classifiers['singleMLP_tfidf-enc'].append(
        experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            encoder=tfidf_encoder,
            model_option='smlp',
            save_to=run_folder,
            smlp_args={
                'hidden_dim': hidden_dim,
                'dropout': dropout,
                'batch_size': batch_size,
                'learning_rate': learning_rate,
                'weight_decay': weight_decay,
                'num_epochs': num_epochs,
            },
            fit_encoder=True
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...


In [ ]:
analysis.plot_box_and_points(analysis.make_data_frame(my_classifiers))

In [ ]:
testing_sentences = test_dataset["sentence"]
testing_labels = test_dataset["label"]

criterion = nn.BCEWithLogitsLoss()

for run, seed in enumerate(my_seeds):
    for fold in range(num_folds):
        model_folder = f"{results_folder}/singleMLP_tfidf-enc/seed-{seed}/fold-{fold}/model"
        idf_path = f"{results_folder}/singleMLP_tfidf-enc/seed-{seed}/fold-{fold}/tfidf_idf.npy"

        tfidf_encoder.load(idf_path)

        testing_vectors = [tfidf_encoder(text) for text in testing_sentences]
        X_test_sparse = csr_matrix(testing_vectors)
        test_dataset_sparse = smlp.SparseDataset(X_test_sparse, testing_labels)

        test_loader = DataLoader(
            test_dataset_sparse,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=smlp.sparse_collate
        )

        model = smlp.load(model_folder)
        test_metrics, preds = smlp.evaluate(model, test_loader, criterion, device)

        print(f"Seed {seed}, fold {fold}:")
        print(test_metrics)
        print(classification_report(testing_labels, preds))

In [8]:
# save results
Path("metrics").mkdir(exist_ok=True)

df = analysis.make_data_frame(my_classifiers)
df["seed"] = df["run"].map(dict(enumerate(my_seeds)))

df.to_csv(f"metrics/{next(iter(my_classifiers))}.csv", index=False)